In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Вкажіть шлях до вашої папки з даними на Google Drive
# Наприклад, якщо ви зберегли папку "vesuvius_data" в корені My Drive:
DATA_DIR = "/content/drive/MyDrive/subit_ink"
FRAGMENT_PATH = os.path.join(DATA_DIR, "train/1")  # або "train/1", залежно від структури

# Перевіряємо, чи все на місці
if os.path.exists(FRAGMENT_PATH):
    print(f"✅ Дані знайдено: {FRAGMENT_PATH}")
    vol_dir = os.path.join(FRAGMENT_PATH, "surface_volume")
    tif_files = [f for f in os.listdir(vol_dir) if f.endswith('.tif')]
    print(f"  Зрізів: {len(tif_files)}")
    print(f"  Маска: {'є' if os.path.exists(os.path.join(FRAGMENT_PATH, 'inklabels.png')) else 'немає'}")
else:
    print(f"❌ Шлях {FRAGMENT_PATH} не існує!")
    print("Перевірте, чи правильно вказано шлях.")
    print("Вміст /content/drive/MyDrive/:")
    for item in os.listdir("/content/drive/MyDrive/"):
        print(f"  - {item}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Дані знайдено: /content/drive/MyDrive/subit_ink/train/1
  Зрізів: 65
  Маска: є


In [3]:
"""
SUBIT-Ink v8 — Повний код з копіюванням даних з Google Drive
===============================================================
Дані: /content/drive/MyDrive/subit_ink/train/1
"""

# ─── ВСТАНОВЛЕННЯ ЗАЛЕЖНОСТЕЙ ─────────────────────────────
import subprocess, sys, os, glob, copy, random, time, zipfile, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from typing import Tuple, List

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Встановлення залежностей...")
install("torch")
install("torchvision")
install("tqdm")
install("pillow")
install("pandas")
install("numpy")
install("scikit-image")
print("✅ Залежності встановлено")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ─── МОНТУВАННЯ GOOGLE DRIVE ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/subit_ink"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Google Drive: {DRIVE_DIR}")

# ─── ПЕРЕВІРКА ТА КОПІЮВАННЯ ДАНИХ ──────────────────────
# Шлях до даних на Google Drive (ваш актуальний шлях)
DRIVE_DATA_PATH = "/content/drive/MyDrive/subit_ink/train/1"
LOCAL_DATA_DIR = "/content/data"
LOCAL_FRAGMENT_PATH = os.path.join(LOCAL_DATA_DIR, "1")

# Перевіряємо, чи дані вже скопійовані локально
if os.path.exists(LOCAL_FRAGMENT_PATH) and len(os.listdir(LOCAL_FRAGMENT_PATH)) > 0:
    print(f"✅ Дані вже скопійовані в {LOCAL_FRAGMENT_PATH}")
else:
    print(f"📦 Копіюємо дані з Google Drive в локальне сховище...")
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    # Копіюємо всю папку
    !cp -r "{DRIVE_DATA_PATH}" "{LOCAL_DATA_DIR}/"
    print(f"✅ Дані скопійовано в {LOCAL_FRAGMENT_PATH}")

# Перевіряємо, що все на місці
if not os.path.exists(LOCAL_FRAGMENT_PATH):
    raise SystemExit(f"❌ Не вдалося скопіювати дані! Перевірте шлях: {DRIVE_DATA_PATH}")

vol_dir = os.path.join(LOCAL_FRAGMENT_PATH, "surface_volume")
if not os.path.exists(vol_dir):
    raise SystemExit(f"❌ surface_volume не знайдено в {LOCAL_FRAGMENT_PATH}")

tif_files = sorted(glob.glob(os.path.join(vol_dir, "*.tif")))
mask_path = os.path.join(LOCAL_FRAGMENT_PATH, "inklabels.png")

print(f"\n✅ Дані готові:")
print(f"  Шлях: {LOCAL_FRAGMENT_PATH}")
print(f"  Зрізів: {len(tif_files)}")
print(f"  Маска: {'є' if os.path.exists(mask_path) else 'немає'}")

# ─── ПЕРЕВІРКА GPU ──────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n🔍 Пристрій: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"  Вільна пам'ять: {free/1024**3:.1f} GB / {total/1024**3:.1f} GB")
else:
    print("  ⚠️ GPU не доступний! Навчання буде повільним.")
    print("  Увімкніть GPU: Runtime → Change runtime type → GPU")

# ─── КОНФІГУРАЦІЯ ──────────────────────────────────────────
CFG = {
    "base_path": LOCAL_DATA_DIR,
    "train_frags": [LOCAL_FRAGMENT_PATH],
    "val_frags": [],
    "test_frags": [],
    "z_start": 27,
    "z_end": 40,
    "in_channels": 13,
    "crop_size": (256, 256),
    "stride_train": (192, 192),
    "stride_val": (128, 128),
    "epochs": 30,
    "batch_size": 4,
    "lr": 3e-5,
    "pos_weight": 3.0,
    "early_stop": 6,
    "threshold": 0.5,
    "tta": True,
    "infer_stride": 64,
    "num_workers": 0,
}

SAVE_PATH = f"{DRIVE_DIR}/subit_ink_best.pt"
print(f"\n📊 Конфігурація:")
print(f"  Train: {CFG['train_frags']}")
print(f"  Зрізи: {CFG['z_start']}–{CFG['z_end']} ({CFG['in_channels']} каналів)")
print(f"  Epochs: {CFG['epochs']}, Batch: {CFG['batch_size']}")
print(f"  Модель зберігається: {SAVE_PATH}")

# ─── UTILS ──────────────────────────────────────────────────
def normalize_patch(patch: np.ndarray) -> np.ndarray:
    p = patch.astype(np.float32) / 255.0
    for c in range(p.shape[0]):
        mn, mx = p[c].min(), p[c].max()
        if mx > mn:
            p[c] = (p[c] - mn) / (mx - mn)
    return p

def f05_score(pred: np.ndarray, target: np.ndarray) -> float:
    tp = float((pred * target).sum())
    fp = float((pred * (1 - target)).sum())
    fn = float(((1 - pred) * target).sum())
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    return (1.25 * prec * rec) / (0.25 * prec + rec + 1e-8)

def mask_to_rle(mask: np.ndarray) -> str:
    pixels = mask.flatten(order='F')
    runs = []
    start = None
    for i, v in enumerate(pixels):
        if v == 1 and start is None:
            start = i + 1
        elif v == 0 and start is not None:
            runs += [start, i - start + 1]
            start = None
    if start is not None:
        runs += [start, len(pixels) - start + 1]
    return ' '.join(map(str, runs)) if runs else ''

# ─── DATASET ────────────────────────────────────────────────
class VesuviusInkDataset(Dataset):
    def __init__(self, fragment_paths, crop_size=(256, 256), stride=(192, 192),
                 z_start=27, z_end=40, augment=True, balance=True,
                 shuffle_chunks=True):
        self.crop_size = crop_size
        self.augment = augment
        self.fragments = []

        for frag_path in fragment_paths:
            vol_dir = os.path.join(frag_path, "surface_volume")
            mask_path = os.path.join(frag_path, "inklabels.png")

            if not os.path.exists(vol_dir):
                print(f"⚠ Пропускаємо {frag_path}: surface_volume не знайдено")
                continue

            # Маска
            if os.path.exists(mask_path):
                try:
                    mask = np.array(Image.open(mask_path))
                    if len(mask.shape) == 3:
                        mask = mask[:, :, 0]
                    mask = (mask > 0).astype(np.uint8)
                except Exception as e:
                    print(f"⚠ Помилка маски: {e}")
                    mask = None
            else:
                mask = None

            # Зрізи
            z_files = sorted(glob.glob(os.path.join(vol_dir, "*.tif")))
            if not z_files:
                print(f"⚠ Немає зрізів в {vol_dir}")
                continue
            z_files = z_files[z_start:z_end]

            slices = []
            for f in z_files:
                try:
                    img = np.array(Image.open(f))
                    if len(img.shape) == 3:
                        img = img[:, :, 0]
                    if img.dtype == np.uint16:
                        img = (img >> 8).astype(np.uint8)
                    else:
                        img = img.astype(np.uint8)
                    slices.append(img)
                except:
                    if slices:
                        slices.append(np.zeros_like(slices[0]))

            if not slices:
                continue

            vol = np.stack(slices, axis=0)
            H, W = vol.shape[1], vol.shape[2]

            if mask is None:
                mask = np.zeros((H, W), dtype=np.uint8)
            elif mask.shape != (H, W):
                from skimage.transform import resize
                mask = (resize(mask, (H, W), preserve_range=True) > 0.5).astype(np.uint8)

            self.fragments.append((vol, mask, H, W))
            print(f"  {os.path.basename(frag_path)}: {vol.shape}, ink={mask.mean():.4f}")

        if not self.fragments:
            raise ValueError("Жоден фрагмент не завантажено!")

        # Чанки
        ink_c, empty_c = [], []
        ch, cw = crop_size
        for fi, (_, mask, H, W) in enumerate(self.fragments):
            for y in range(0, H - ch + 1, stride[0]):
                for x in range(0, W - cw + 1, stride[1]):
                    entry = (fi, y, x)
                    if mask[y:y+ch, x:x+cw].sum() > 100:
                        ink_c.append(entry)
                    else:
                        empty_c.append(entry)

        if shuffle_chunks:
            random.shuffle(ink_c)
            random.shuffle(empty_c)

        if balance and ink_c:
            self.chunks = ink_c + empty_c[:len(ink_c)]
        else:
            self.chunks = ink_c + empty_c

        if shuffle_chunks:
            random.shuffle(self.chunks)

        print(f"  → {len(self.chunks)} чанків ({len(ink_c)} ink + {len(empty_c)} empty)")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        fi, y, x = self.chunks[idx]
        vol, mask, H, W = self.fragments[fi]
        h, w = self.crop_size
        y2, x2 = min(y + h, H), min(x + w, W)

        ct = vol[:, y:y2, x:x2]
        msk = mask[y:y2, x:x2].astype(np.int64)

        if ct.shape[1] < h or ct.shape[2] < w:
            ph, pw = h - ct.shape[1], w - ct.shape[2]
            ct = np.pad(ct, ((0, 0), (0, ph), (0, pw)), mode='reflect')
            msk = np.pad(msk, ((0, ph), (0, pw)), mode='reflect')

        ct = normalize_patch(ct)

        # Аугментація
        if self.augment:
            if np.random.rand() > 0.5:
                ct = np.flip(ct, 1).copy()
                msk = np.flip(msk, 0).copy()
            if np.random.rand() > 0.5:
                ct = np.flip(ct, 2).copy()
                msk = np.flip(msk, 1).copy()
            k = np.random.randint(0, 4)
            if k:
                ct = np.rot90(ct, k, (1, 2)).copy()
                msk = np.rot90(msk, k, (0, 1)).copy()

        return torch.from_numpy(ct), torch.from_numpy(msk)

# ─── МОДЕЛІ ─────────────────────────────────────────────────
class CurvatureField2D(nn.Module):
    def __init__(self, sigma=1.0):
        super().__init__()
        size = int(6 * sigma + 1) | 1
        coords = torch.arange(size).float() - size // 2
        g = torch.exp(-coords**2 / (2 * sigma**2))
        g = g / g.sum()
        k = (g[:, None] * g[None, :]).unsqueeze(0).unsqueeze(0)
        self.smooth = nn.Conv2d(1, 1, size, padding=size//2, bias=False)
        self.smooth.weight = nn.Parameter(k, requires_grad=False)

    def _d2(self, x, dim):
        pad = (1, 1, 0, 0) if dim == 3 else (0, 0, 1, 1)
        xp = F.pad(x, pad, mode='replicate')
        s = [slice(None)] * xp.dim()
        sm, sc, sp = list(s), list(s), list(s)
        sm[dim] = slice(0, -2)
        sc[dim] = slice(1, -1)
        sp[dim] = slice(2, None)
        return xp[tuple(sm)] - 2 * xp[tuple(sc)] + xp[tuple(sp)]

    def forward(self, x):
        xs = self.smooth(x.mean(1, keepdim=True))
        hyy, hxx = self._d2(xs, 2), self._d2(xs, 3)
        l1 = torch.min(hxx.abs(), hyy.abs())
        l2 = torch.max(hxx.abs(), hyy.abs())
        n = l2 + 1e-6
        return torch.cat([
            1 - l1 / n,
            l1 / n,
            ((hxx < 0) & (hyy < 0)).float(),
            ((hxx > 0) & (hyy > 0)).float()
        ], dim=1)

class SUBITEncoder2D(nn.Module):
    SCALES = [1, 2, 4, 8]
    def __init__(self, edim=32):
        super().__init__()
        self.fields = nn.ModuleList([CurvatureField2D(s) for s in self.SCALES])
        self.proj = nn.Sequential(
            nn.Conv2d(16, edim//2, 1),
            nn.GroupNorm(8, edim//2),
            nn.GELU(),
            nn.Conv2d(edim//2, edim, 1)
        )
    def forward(self, x):
        return self.proj(torch.cat([f(x) for f in self.fields], dim=1))

class Ftheta2D(nn.Module):
    def __init__(self, ch, steps=3):
        super().__init__()
        self.steps = steps
        self.diff = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, groups=ch),
            nn.Conv2d(ch, ch, 1),
            nn.GroupNorm(min(8, ch), ch),
            nn.GELU(),
            nn.Conv2d(ch, 2, 1)
        )
        self.dt = nn.Parameter(torch.ones(1, ch, 1, 1) * 0.1)
        self.proj = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.GroupNorm(min(8, ch), ch),
            nn.GELU()
        )
        self.gate = nn.Sequential(nn.Conv2d(ch, 1, 1), nn.Sigmoid())

    def _step(self, x, D):
        Dxx, Dyy = D[:, 0:1].abs(), D[:, 1:2].abs()
        xp = nn.ReplicationPad2d(1)(x)
        d2y = xp[:, :, 2:, 1:-1] - 2 * x + xp[:, :, :-2, 1:-1]
        d2x = xp[:, :, 1:-1, 2:] - 2 * x + xp[:, :, 1:-1, :-2]
        return x + self.dt * (Dyy * d2y + Dxx * d2x)

    def forward(self, x):
        r = x
        D = self.diff(x)
        g = self.gate(x)
        h = x
        for _ in range(self.steps):
            h = self._step(h, D)
        return r + g * (self.proj(h) - r)

class Block2D(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1),
            nn.GroupNorm(min(8, oc), oc),
            nn.GELU(),
            nn.Conv2d(oc, oc, 3, padding=1),
            nn.GroupNorm(min(8, oc), oc),
            nn.GELU()
        )
        self.s = nn.Conv2d(ic, oc, 1) if ic != oc else nn.Identity()
    def forward(self, x):
        return self.b(x) + self.s(x)

class Attn2D(nn.Module):
    def __init__(self, fc, sc):
        super().__init__()
        self.g = nn.Sequential(
            nn.Conv2d(fc + sc, fc, 1),
            nn.GroupNorm(min(8, fc), fc),
            nn.GELU(),
            nn.Conv2d(fc, fc, 1),
            nn.Sigmoid()
        )
    def forward(self, f, s):
        return f * self.g(torch.cat([f, s], dim=1))

class SUBITInkUNet(nn.Module):
    def __init__(self, in_ch=13, features=(64, 128, 256, 512), edim=32):
        super().__init__()
        self.subit = SUBITEncoder2D(edim)
        self.inp = nn.Conv2d(in_ch + edim, features[0], 1)
        self.encs = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.eattn = nn.ModuleList()
        ch = features[0]
        for f in features:
            self.encs.append(Block2D(ch, f))
            self.pools.append(nn.MaxPool2d(2))
            self.eattn.append(Attn2D(f, edim))
            ch = f
        self.btn = Block2D(features[-1], features[-1] * 2)
        self.evol = Ftheta2D(features[-1] * 2, steps=3)
        self.ups = nn.ModuleList()
        self.decs = nn.ModuleList()
        self.dattn = nn.ModuleList()
        ch = features[-1] * 2
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(ch, f, 2, stride=2))
            self.decs.append(Block2D(f * 2, f))
            self.dattn.append(Attn2D(f, edim))
            ch = f
        self.out = nn.Conv2d(features[0], features[0], 1)

    def forward(self, x):
        se = self.subit(x)
        ss = [se]
        for _ in range(len(self.encs) - 1):
            ss.append(F.avg_pool2d(ss[-1], 2))
        h = self.inp(torch.cat([x, se], dim=1))
        skips = []
        for i, (enc, pool, attn) in enumerate(zip(self.encs, self.pools, self.eattn)):
            h = attn(enc(h), ss[i])
            skips.append(h)
            h = pool(h)
        h = self.evol(self.btn(h))
        for up, dec, attn, skip in zip(self.ups, self.decs, self.dattn, reversed(skips)):
            h = up(h)
            if h.shape != skip.shape:
                h = F.interpolate(h, size=skip.shape[2:], mode='bilinear', align_corners=False)
            h = attn(dec(torch.cat([h, skip], dim=1)),
                     F.interpolate(se, size=h.shape[2:], mode='bilinear', align_corners=False))
        return self.out(h), se

class AttractorDecoder2D(nn.Module):
    def __init__(self, ch=64, steps=5):
        super().__init__()
        self.steps = steps
        self.ftheta = Ftheta2D(ch, steps=1)
        self.omega = nn.Sequential(
            nn.Conv2d(ch * 2, ch, 1),
            nn.GroupNorm(min(8, ch), ch),
            nn.GELU(),
            nn.Conv2d(ch, 4, 1),
            nn.Softmax(dim=1)
        )
        self.head = nn.Conv2d(ch, 2, 1)

    def forward(self, x):
        h = x
        delta = torch.zeros_like(x)
        for _ in range(self.steps):
            hn = self.ftheta(h)
            delta += (hn - h).abs()
            h = hn
        omega = self.omega(torch.cat([h, delta], dim=1))
        w = omega[:, 3:4] * 1.5 + omega[:, 2:3] * 1.2 + omega[:, 1:2] * 0.8 + omega[:, 0:1] * 0.3
        return self.head(h * w), omega

# ─── LOSS ───────────────────────────────────────────────────
class OmegaLoss2D(nn.Module):
    def __init__(self, pos_weight=3.0, alpha=0.4, beta=0.3,
                 gamma_l=0.2, delta=0.1, focal_gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma_l = gamma_l
        self.delta = delta
        self.fg = focal_gamma
        self.register_buffer('w', torch.tensor([1.0, pos_weight]))

    def focal(self, logits, labels):
        ce = F.cross_entropy(logits, labels, weight=self.w.to(logits.device), reduction='none')
        return ((1 - torch.exp(-ce)) ** self.fg * ce).mean()

    def forward(self, logits, labels, omega, se):
        probs = F.softmax(logits, dim=1)
        fl = self.focal(logits, labels)
        smooth = 1e-5
        p, l = probs[:, 1], labels.float()
        inter = (p * l).sum(dim=(1, 2))
        union = p.sum(dim=(1, 2)) + l.sum(dim=(1, 2))
        dice = (1 - (2 * inter + smooth) / (union + smooth)).mean()
        if se.shape[2:] != probs.shape[2:]:
            se = F.interpolate(se, size=probs.shape[2:], mode='bilinear', align_corners=False)
        curv = (torch.sigmoid(se[:, 0]) * (1 - probs[:, 1])).mean()
        stab = (omega[:, 0] * probs[:, 1]).mean()
        om = omega.mean(dim=(0, 2, 3))
        oreg = (om * (om + 1e-8).log()).sum() * 0.01
        total = (self.alpha * fl + (1 - self.alpha) * dice +
                 self.gamma_l * curv + self.beta * stab + self.delta * oreg)
        return total, {
            "focal": fl.item(),
            "dice": dice.item(),
            "curv": curv.item(),
            "stab": stab.item(),
            "total": total.item()
        }

# ─── PIPELINE ──────────────────────────────────────────────
class InkPipeline:
    def __init__(self, in_ch=13, pos_weight=3.0, lr=3e-5, device='cuda'):
        self.device = device
        self.model = SUBITInkUNet(in_ch=in_ch).to(device)
        self.attractor = AttractorDecoder2D(ch=64).to(device)
        self.loss_fn = OmegaLoss2D(pos_weight=pos_weight)
        params = list(self.model.parameters()) + list(self.attractor.parameters())
        self.optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-5)
        self.use_amp = device == 'cuda'
        if self.use_amp:
            self.scaler = torch.amp.GradScaler('cuda')

    def _run(self, ct, mask):
        x, y = ct.to(self.device), mask.to(self.device)
        if self.use_amp:
            with torch.amp.autocast('cuda'):
                feats, se = self.model(x)
                logits, omega = self.attractor(feats)
                if logits.shape[2:] != y.shape[1:]:
                    logits = F.interpolate(logits, size=y.shape[1:], mode='bilinear', align_corners=False)
                    omega = F.interpolate(omega, size=y.shape[1:], mode='bilinear', align_corners=False)
                loss, comps = self.loss_fn(logits, y, omega, se)
        else:
            feats, se = self.model(x)
            logits, omega = self.attractor(feats)
            if logits.shape[2:] != y.shape[1:]:
                logits = F.interpolate(logits, size=y.shape[1:], mode='bilinear', align_corners=False)
                omega = F.interpolate(omega, size=y.shape[1:], mode='bilinear', align_corners=False)
            loss, comps = self.loss_fn(logits, y, omega, se)
        with torch.no_grad():
            pred = (F.softmax(logits, dim=1)[:, 1] > 0.5).long()
            p, t = pred.float().flatten(), y.float().flatten()
            comps["dice_m"] = ((2 * (p * t).sum() + 1e-5) / (p.sum() + t.sum() + 1e-5)).item()
        return loss, comps

    def train_step(self, ct, mask):
        self.model.train()
        self.attractor.train()
        self.optimizer.zero_grad()
        loss, comps = self._run(ct, mask)
        if self.use_amp:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(self.model.parameters()) + list(self.attractor.parameters()), 0.5)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.model.parameters()) + list(self.attractor.parameters()), 0.5)
            self.optimizer.step()
        return comps

    def val_step(self, ct, mask):
        self.model.eval()
        self.attractor.eval()
        with torch.no_grad():
            _, comps = self._run(ct, mask)
        return comps

    def predict_probs(self, ct_np):
        x = torch.from_numpy(ct_np).unsqueeze(0).to(self.device)
        if self.use_amp:
            with torch.no_grad(), torch.amp.autocast('cuda'):
                feats, _ = self.model(x)
                logits, _ = self.attractor(feats)
        else:
            with torch.no_grad():
                feats, _ = self.model(x)
                logits, _ = self.attractor(feats)
        return F.softmax(logits, dim=1)[0, 1].float().cpu().numpy()

    def predict_tta(self, ct_np):
        self.model.eval()
        self.attractor.eval()
        p = self.predict_probs(ct_np)
        p += self.predict_probs(ct_np[:, ::-1, :].copy())[::-1, :]
        p += self.predict_probs(ct_np[:, :, ::-1].copy())[:, ::-1]
        p += self.predict_probs(ct_np[:, ::-1, ::-1].copy())[::-1, ::-1]
        return p / 4.0

    def save(self, path, meta=None):
        torch.save({
            'model': self.model.state_dict(),
            'attractor': self.attractor.state_dict(),
            'meta': meta or {}
        }, path)
        print(f"  ✅ Збережено: {path}")

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt['model'])
        self.attractor.load_state_dict(ckpt['attractor'])
        meta = ckpt.get('meta', {})
        print(f"  ✅ Завантажено: {path}")
        if meta:
            print(f"    meta: {meta}")
        return meta

# ─── TRAINING LOOP ──────────────────────────────────────────
def train_loop(pipeline, train_loader, val_loader, epochs, save_path, patience=6):
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        pipeline.optimizer, T_max=epochs, eta_min=1e-6)
    best, no_improve = 0.0, 0

    for ep in range(epochs):
        tl, td = 0.0, 0.0
        pbar = tqdm(train_loader, desc=f"Train {ep+1}/{epochs}")
        for ct, mask in pbar:
            m = pipeline.train_step(ct, mask)
            tl += m['total']
            td += m['dice_m']
            pbar.set_postfix({'L': f"{m['total']:.3f}", 'D': f"{m['dice_m']:.3f}"})

        vl, vd = 0.0, 0.0
        for ct, mask in val_loader:
            m = pipeline.val_step(ct, mask)
            vl += m['total']
            vd += m['dice_m']

        nt, nv = len(train_loader), len(val_loader)
        avg_vd = vd / nv
        print(f"E{ep+1:02d}: TL={tl/nt:.4f} TD={td/nt:.4f} | "
              f"VL={vl/nv:.4f} VD={avg_vd:.4f} | LR={sched.get_last_lr()[0]:.1e}")
        sched.step()

        if avg_vd > best:
            best = avg_vd
            no_improve = 0
            pipeline.save(save_path, meta={'epoch': ep+1, 'val_dice': best})
            print(f"  ✅ Best Val Dice={best:.4f}")
        else:
            no_improve += 1
            print(f"  (no improve {no_improve}/{patience})")
            if no_improve >= patience:
                print(f"  Early stopping на E{ep+1}")
                break
    return best

# ─── ОСНОВНИЙ ПАЙПЛАЙН ─────────────────────────────────────
print("\n" + "="*60)
print("🚀 ЗАПУСК НАВЧАННЯ")
print("="*60)

# Створюємо датасети
print("\n[1/3] Створення датасетів...")
train_ds = VesuviusInkDataset(
    CFG['train_frags'],
    crop_size=CFG['crop_size'],
    stride=CFG['stride_train'],
    z_start=CFG['z_start'],
    z_end=CFG['z_end'],
    augment=True,
    balance=True
)

# Валідаційний набір (20% від train)
val_ds = copy.copy(train_ds)
n_val = max(1, int(len(train_ds) * 0.2))
val_ds.chunks = train_ds.chunks[-n_val:]
train_ds.chunks = train_ds.chunks[:-n_val]
val_ds.augment = False

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=CFG['num_workers'])
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'],
                        shuffle=False, num_workers=CFG['num_workers'])

print(f"Train чанків: {len(train_ds)}")
print(f"Val чанків:   {len(val_ds)}")

# Ініціалізація моделі
print("\n[2/3] Ініціалізація моделі...")
pipeline = InkPipeline(
    in_ch=CFG['in_channels'],
    pos_weight=CFG['pos_weight'],
    lr=CFG['lr'],
    device=device
)
print(f"Параметрів моделі: {sum(p.numel() for p in pipeline.model.parameters()):,}")

# Навчання
print("\n[3/3] Запуск навчання...")
best_dice = train_loop(
    pipeline,
    train_loader,
    val_loader,
    epochs=CFG['epochs'],
    save_path=SAVE_PATH,
    patience=CFG['early_stop']
)

print(f"\n🏁 Найкращий Val Dice: {best_dice:.4f}")
print(f"✅ Модель збережена: {SAVE_PATH}")
print("\n🎉 Тренування завершено!")

Встановлення залежностей...
✅ Залежності встановлено
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive: /content/drive/MyDrive/subit_ink
📦 Копіюємо дані з Google Drive в локальне сховище...
✅ Дані скопійовано в /content/data/1

✅ Дані готові:
  Шлях: /content/data/1
  Зрізів: 65
  Маска: є

🔍 Пристрій: cuda
  GPU: Tesla T4
  Вільна пам'ять: 14.5 GB / 14.6 GB

📊 Конфігурація:
  Train: ['/content/data/1']
  Зрізи: 27–40 (13 каналів)
  Epochs: 30, Batch: 4
  Модель зберігається: /content/drive/MyDrive/subit_ink/subit_ink_best.pt

🚀 ЗАПУСК НАВЧАННЯ

[1/3] Створення датасетів...
  1: (13, 8181, 6330), ink=0.1031
  → 958 чанків (479 ink + 865 empty)
Train чанків: 767
Val чанків:   191

[2/3] Ініціалізація моделі...
Параметрів моделі: 44,453,311

[3/3] Запуск навчання...


Train 1/30: 100%|██████████| 192/192 [02:10<00:00,  1.47it/s, L=0.668, D=0.358]


E01: TL=0.6875 TD=0.3397 | VL=0.6640 VD=0.3959 | LR=3.0e-05
  ✅ Збережено: /content/drive/MyDrive/subit_ink/subit_ink_best.pt
  ✅ Best Val Dice=0.3959


Train 2/30: 100%|██████████| 192/192 [02:02<00:00,  1.57it/s, L=0.737, D=0.045]


E02: TL=0.6710 TD=0.3497 | VL=0.6605 VD=0.3945 | LR=3.0e-05
  (no improve 1/6)


...

E09: TL=0.6525 TD=0.3581 | VL=0.6438 VD=0.3943 | LR=2.5e-05
  (no improve 4/6)


Train 10/30: 100%|██████████| 192/192 [02:02<00:00,  1.57it/s, L=0.711, D=0.000]


E10: TL=0.6507 TD=0.3677 | VL=0.6444 VD=0.3956 | LR=2.4e-05
  (no improve 5/6)


Train 11/30: 100%|██████████| 192/192 [02:02<00:00,  1.57it/s, L=0.615, D=0.438]


E11: TL=0.6498 TD=0.3633 | VL=0.6405 VD=0.3940 | LR=2.3e-05
  (no improve 6/6)
  Early stopping на E11

🏁 Найкращий Val Dice: 0.3961
✅ Модель збережена: /content/drive/MyDrive/subit_ink/subit_ink_best.pt

🎉 Тренування завершено!
